# C — Error Analyzer có tuỳ chọn abstention

**Đóng ý kiến #6 / Stanford Q5:** bộ phân tích hiện sụp về 2/5 lớp (κ = 0,16; không bao giờ
dự đoán *calculation* hay *careless*). Chính bài kết luận "an abstention option is needed".

Notebook này chạy lại **đúng pipeline Error Analyzer đã công bố** (notebook tái lập gốc, cell 32–33) trên
đúng 46 lượt CoMTA, với **một thay đổi duy nhất: thêm lớp `unclear`** vào prompt và validator
(hai chỗ đánh dấu `[+unclear]`). Giữ nguyên: system prompt, văn bản prompt, đầu vào (history, KC,
mastery QLoRA seed 221, reference/extracted), validator JSON, rule-based fallback, Llama-3.1-8B-Instruct
4-bit NF4 double-quant, greedy, batch 8 left-padding, 320 token, thứ tự lượt.

**Tự kiểm trước khi chạy mô hình:** (1) công thức metric tái lập đúng κ/F1/exact đã công bố từ file
predictions gốc; (2) đầu vào dựng lại cho ra đúng `category_rule_based` 46/46 và severity của 4 lượt fallback.

**Cần:** GPU Colab (T4) + HF token + thư mục `revision_kit` trên MyDrive.

In [ ]:
!pip -q install transformers accelerate bitsandbytes scikit-learn
from huggingface_hub import notebook_login; notebook_login()

In [ ]:
import json, os, re, torch, pandas as pd, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import cohen_kappa_score, f1_score

from google.colab import drive; drive.mount("/content/drive")
KIT = "/content/drive/MyDrive/revision_kit"                      # <<< upload nguyen thu muc revision_kit vao MyDrive
MATHKT_ROOT = f"{KIT}/MathKT-Agent_SoICT2026"                   # goi tai lap (da chep san trong revision_kit)
OUT_DIR = f"{KIT}/colab_out"                                    # dau ra ghi thang len Drive (khong mat khi runtime reset)
os.makedirs(OUT_DIR, exist_ok=True); os.chdir(OUT_DIR)
COMTA_JSON  = f"{KIT}/out/comta_context.json"

CLASSES = ["conceptual", "procedural", "calculation", "careless", "other"]
CATEGORIES = CLASSES + ["unclear"]                                # [+unclear]
SEVERITIES = ["low", "medium", "high"]
EA = f"{MATHKT_ROOT}/results/05_error_analyzer"

gold = pd.read_csv(f"{EA}/error_analyzer_gold.csv", dtype={"dialogue": str})
gold["gold"] = gold.gold.astype(str).str.strip().str.lower()
gold["gold_secondary"] = gold.gold_secondary.fillna("").astype(str).str.strip().str.lower()
assert len(gold) == 46 and set(gold.gold) <= set(CLASSES)

def score(pred, labels=CLASSES):
    # Cung cong thuc voi ea_annotation.py cmd_metrics (so da cong bo)
    m = pred.merge(gold, on=["dialogue", "turn_id"])
    p, g, s = m.category, m.gold, m.gold_secondary
    return dict(n=len(m), exact_match=float((p == g).mean()),
                macro_f1=float(f1_score(g, p, labels=CLASSES, average="macro", zero_division=0)),
                cohen_kappa=float(cohen_kappa_score(g, p, labels=labels)),
                relaxed_agreement=float(((p == g) | (p == s)).mean()))

# Tu kiem 1: cong thuc tren phai tai lap dung so da cong bo tu predictions goc
PUB = pd.read_csv(f"{EA}/error_analyzer_predictions.csv", dtype={"dialogue": str})
PUB = PUB[PUB.set == "comta_incorrect_label"].reset_index(drop=True)
pub = score(PUB)
ref = pd.read_csv(f"{EA}/error_analyzer_metrics.csv").set_index("method").loc["llm_error_analyzer"]
for a, b in [("exact_match", "exact_match"), ("macro_f1", "macro_f1"),
             ("cohen_kappa", "cohen_kappa_vs_gold"), ("relaxed_agreement", "relaxed_agreement")]:
    assert abs(pub[a] - ref[b]) < 1e-9, (a, pub[a], ref[b])
print("Tu kiem metric OK:", {k: round(v, 4) for k, v in pub.items()})

### Đầu vào — dựng lại đúng như lần chạy gốc

`comta_context.json` được dựng từ `comta_atc.csv` theo `labelled_turns(skip_first_turn=False, with_context=True)`
(id hội thoại = vị trí dòng; history `Tutor:/Student:`, 2500 ký tự cuối; `problem=None`) và
`export_mastery_lookup` (RUN_QLORA=True, seed 221; 9/46 lượt không có mastery → "not available", như bản gốc).
CoMTA không có đáp án tham chiếu nên SAVA trả `undetermined`, `extracted=None` cho cả 46 lượt.
Prompt gốc và rule-based fallback dưới đây chép nguyên văn từ notebook tái lập.

In [ ]:
def load_context():
    if not os.path.exists(COMTA_JSON):
        raise SystemExit("Chua co comta_context.json. Kiem tra da upload ca thu muc revision_kit (gom out/comta_context.json).")
    data = json.load(open(COMTA_JSON))["contexts"]
    return {(k.split("|")[0], int(k.split("|")[1])): v for k, v in data.items()}
CTX = load_context()

ITEMS = []
for d, t in zip(PUB.dialogue, PUB.turn_id.astype(int)):   # dung thu tu lan chay goc -> cung cach chia batch
    c = CTX[(d, t)]
    assert c["label"] is False, (d, t)
    ITEMS.append(dict(c, dialogue=d, turn_id=t, extracted=None, reference_parsed=None))   # sava_verify(text, None)
assert {(i["dialogue"], i["turn_id"]) for i in ITEMS} == set(zip(gold.dialogue, gold.turn_id.astype(int)))

PEDAGOGY_SYSTEM = "You are a mathematics tutor. Reply with a single JSON object only."

def short_kc(kc, limit=160):
    kc = str(kc)
    return kc if len(kc) <= limit else kc[:limit - 3] + "..."

def error_analyzer_prompt(item):
    mastery = ", ".join(f"{short_kc(k, 60)}: {v:.2f}" for k, v in item["mastery"].items()) or "not available"
    return (
        "Analyse the student's error in a mathematics tutoring dialogue.\n"
        f"Problem: {item.get('problem') or 'not provided'}\n"
        f"Reference answer: {item.get('reference') or 'not provided'}\n"
        f"Dialogue so far:\n{item['history']}\n\n"
        f"Student turn to analyse: {item['student_text']}\n"
        f"Extracted student expression (symbolic verifier): {item.get('extracted')}\n"
        f"Knowledge components involved: {'; '.join(short_kc(k) for k in item['kcs'])}\n"
        f"Current mastery estimates: {mastery}\n\n"
        "Category definitions: conceptual = misunderstanding of a concept or of the relation between quantities; "
        "procedural = a wrong, missing or misordered step of a method; calculation = an arithmetic slip inside an "
        "otherwise correct method; careless = a copying, sign or reading slip; other = none of these; "
        "unclear = the evidence in this turn does not justify any of the categories above.\n"            # [+unclear]
        "Return JSON with exactly these keys: error_category (one of conceptual, procedural, calculation, careless, other, unclear), "  # [+unclear]
        "explanation (string), affected_concepts (list of KC names from the list above), severity (low|medium|high), "
        "instructional_suggestions (list of 2-3 strings)."
    )

def parse_json_object(text):
    match = re.search(r"\{.*\}", str(text), flags=re.S)
    if not match:
        return None
    try:
        value = json.loads(match.group(0))
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None

def validate_error_analysis(obj):
    if not obj:
        return None
    category = str(obj.get("error_category", "")).strip().lower()
    severity = str(obj.get("severity", "")).strip().lower()
    explanation, concepts, suggestions = obj.get("explanation"), obj.get("affected_concepts"), obj.get("instructional_suggestions")
    if category not in CATEGORIES or severity not in SEVERITIES:                                    # [+unclear]
        return None
    if not isinstance(explanation, str) or not explanation.strip():
        return None
    if not isinstance(concepts, list) or not isinstance(suggestions, list) or not suggestions:
        return None
    return {"error_category": category, "explanation": explanation.strip(), "severity": severity,
            "affected_concepts": [str(c) for c in concepts], "instructional_suggestions": [str(s) for s in suggestions]}

def rule_based_error_analysis(item):
    # Port of kse2026_code/pedagogy.py ErrorAnalyzer.rule_based (nguyen van notebook goc)
    text = str(item["student_text"]).strip().lower()
    extracted, reference = item.get("extracted"), item.get("reference_parsed")
    mastery, kcs = item["mastery"], item["kcs"]
    category, explanation = "other", "The error could not be attributed to a specific mechanism from the text alone."
    try:
        a, b = float(extracted), float(reference)
        if abs(a + b) < 1e-9 and b != 0:
            category, explanation = "careless", f"The answer {a:g} has the right magnitude but the wrong sign."
        elif abs(a - b) <= 1:
            category, explanation = "careless", f"The answer {a:g} is off by {abs(a - b):g} from {b:g}."
        elif b != 0 and (abs(a / b - 2) < 1e-9 or abs(a / b - 0.5) < 1e-9):
            category, explanation = "procedural", f"The answer {a:g} is {b:g} scaled by two: a step was applied or omitted."
        elif b != 0 and abs(a * b - 1) < 1e-9:
            category, explanation = "conceptual", f"The answer {a:g} is the reciprocal of {b:g}: the relation was inverted."
        elif b != 0 and abs(a - b) / max(1.0, abs(b)) < 0.15:
            category, explanation = "calculation", f"The answer {a:g} is close to {b:g}: an arithmetic slip."
        else:
            category, explanation = "procedural", f"The answer {a:g} differs substantially from {b:g}."
    except (TypeError, ValueError):
        symbolic_ext = extracted is not None and re.search(r"[a-z]", str(extracted)) is not None
        symbolic_ref = reference is not None and re.search(r"[a-z]", str(reference)) is not None
        if symbolic_ext and symbolic_ref:
            category, explanation = "procedural", "A rule was applied to only part of the expression."
        elif symbolic_ext:
            category, explanation = "procedural", "The student stopped at an intermediate algebraic form."
        elif any(w in text for w in ("because", "since", "means", "should", "rule", "always")):
            category, explanation = "conceptual", "The explanation reveals a misconception about the underlying rule."
    weakest = min((mastery.get(k, 0.5) for k in kcs), default=0.5)
    if category in ("careless", "calculation"):
        severity = "low" if weakest >= 0.5 else "medium"
    else:
        severity = "high" if weakest < 0.4 else "medium"
    affected = [k for k in kcs if mastery.get(k, 0.5) < 0.7] or list(kcs)
    return {"error_category": category, "explanation": explanation, "severity": severity,
            "affected_concepts": affected, "instructional_suggestions": ["Ask the student to explain the step that produced this answer."]}

# Tu kiem 2: dau vao dung lai phai cho ra dung ket qua rule-based da cong bo
for it, r in zip(ITEMS, PUB.itertuples()):
    rb = rule_based_error_analysis(it)
    assert json.loads(r.kcs) == it["kcs"], (r.dialogue, r.turn_id)
    assert rb["error_category"] == r.category_rule_based, (r.dialogue, r.turn_id)
    if r.source == "rule_fallback":
        assert rb["severity"] == r.severity, (r.dialogue, r.turn_id)
print("Tu kiem dau vao OK:", len(ITEMS), "luot;", sum(bool(i["mastery"]) for i in ITEMS), "co mastery")

In [ ]:
# Nap mo hinh dung nhu load_generator() goc (PEDAGOGY_USE_KT_ADAPTER=False: base instruct)
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
GEN_BATCH_SIZE, GEN_MAX_NEW_TOKENS = 8, 320
GEN_TOKENIZER = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side="left")
GEN_TOKENIZER.pad_token = GEN_TOKENIZER.eos_token
GEN_MODEL = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, dtype=torch.float16, device_map={"": 0},
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                                           bnb_4bit_compute_dtype=torch.float16))
GEN_MODEL.eval()

def generate_texts_local(prompts):
    outputs = []
    for start in range(0, len(prompts), GEN_BATCH_SIZE):
        batch = prompts[start:start + GEN_BATCH_SIZE]
        chats = [GEN_TOKENIZER.apply_chat_template([{"role": "system", "content": PEDAGOGY_SYSTEM},
                                                    {"role": "user", "content": p}],
                                                   tokenize=False, add_generation_prompt=True) for p in batch]
        encoded = GEN_TOKENIZER(chats, return_tensors="pt", padding=True, add_special_tokens=False).to(GEN_MODEL.device)
        with torch.inference_mode():
            generated = GEN_MODEL.generate(**encoded, max_new_tokens=GEN_MAX_NEW_TOKENS, do_sample=False,
                                           temperature=None, top_p=None, pad_token_id=GEN_TOKENIZER.pad_token_id)
        for row in generated[:, encoded["input_ids"].shape[1]:]:
            outputs.append(GEN_TOKENIZER.decode(row, skip_special_tokens=True))
        print(f"generated {len(outputs)}/{len(prompts)}")
    return outputs

### Chạy Error Analyzer (JSON không hợp lệ → rule-based fallback, như bản gốc)

In [ ]:
outputs = generate_texts_local([error_analyzer_prompt(it) for it in ITEMS])
rows = []
for it, text in zip(ITEMS, outputs):
    parsed = validate_error_analysis(parse_json_object(text))
    rule = rule_based_error_analysis(it)
    final = parsed or rule
    rows.append(dict(dialogue=it["dialogue"], turn_id=it["turn_id"], valid_json=parsed is not None,
                     source="llm" if parsed else "rule_fallback", category=final["error_category"],
                     severity=final["severity"], explanation=final["explanation"],
                     category_rule_based=rule["error_category"], raw_output=str(text)[:4000]))
P = pd.DataFrame(rows); P.to_csv("ea_abstention_predictions.csv", index=False)
print(P.source.value_counts().to_dict()); print(P.category.value_counts().to_dict())

### Chấm điểm

Công thức giống `ea_annotation.py`; báo cả hai thang:
- **all_abstain_as_error** (n = 46): `unclear` tính là sai — so trực tiếp với số đã công bố.
  κ tính trên 6 nhãn để lượt `unclear` **không** bị sklearn âm thầm loại khỏi ma trận nhầm lẫn.
- **judged_only**: bỏ các lượt `unclear` — độ chính xác trên phần mô hình dám trả lời.

In [ ]:
J = P[P.category != "unclear"]
res = dict(abstention_rate=float((P.category == "unclear").mean()),
           valid_json_rate=float(P.valid_json.mean()),
           all_abstain_as_error=score(P, labels=CATEGORIES),
           judged_only=score(J),
           published_no_abstention=pub)
json.dump(res, open("ea_abstention_metrics.json", "w"), indent=1)
print(json.dumps(res, indent=1))
M = P.merge(gold, on=["dialogue", "turn_id"])
print("\nConfusion (rows gold, columns LLM):")
print(pd.crosstab(M.gold, M.category).reindex(index=CLASSES, columns=CATEGORIES, fill_value=0))
print("\n-> ea_abstention_predictions.csv, ea_abstention_metrics.json")